# Dr.Retina — P1 (augmented ResNet-50), T4-maximized

Run this in a **second Google account** in parallel with the existing P1 run, using a config tuned to actually use the T4's 16GB (the checked-in `p1_augmented_colab.yaml` used `batch_size=16`, chosen conservatively without measuring headroom). This notebook:

1. Mounts Drive for persistent checkpoints (survives disconnects, same as the original P1 run)
2. Installs dependencies
3. Extracts the uploaded project zip
4. **Probes the actual max batch size that fits in the T4's memory** and writes it into the config
5. Launches training with `--resume` support

If this run finishes a comparable number of epochs faster and its internal-val QWK is at least as good, use its checkpoint instead of (or to compare against) the original P1 run — see `docs/SIH_SUBMISSION_PLAN.md` item 1's selection rule (pick on internal QWK only, no peeking at external results).

## 0. Sanity-check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 1. Mount Drive (for checkpoints that survive a Colab disconnect)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Upload the project zip

Same zip package used for the original P1 run (project source + `data/processed` + `data/splits`). If it's already in Drive, skip the upload cell and just point `PROJECT_ZIP` at the Drive path instead.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select dr_retina_project.zip
PROJECT_ZIP = next(iter(uploaded.keys()))

In [ ]:
# If instead loading from Drive, comment out the upload cell above and uncomment this:
# PROJECT_ZIP = '/content/drive/MyDrive/dr_retina_project.zip'

In [ ]:
import zipfile, os
os.makedirs('/content/dr_retina', exist_ok=True)
with zipfile.ZipFile(PROJECT_ZIP, 'r') as zf:
    zf.extractall('/content/dr_retina')
%cd /content/dr_retina
!ls

## 3. Install dependencies

`torch`/`torchvision` already ship with the Colab runtime — only the project's extra deps need installing. `opencv-python-headless` avoids a GUI-lib dependency error that plain `opencv-python` sometimes hits on Colab.

In [ ]:
!pip install -q timm>=0.9.16 albumentations>=1.4.0 opencv-python-headless>=4.9.0 \
    scikit-learn>=1.4.0 scikit-image>=0.22.0 pyyaml>=6.0.1

## 4. Critical env var — set *before* anything imports albumentations

albumentations does a PyPI version-check on import that can hang for 2-40s+ per attempt on a flaky connection, and DataLoader workers re-import it on every spawn — this silently ate most of a day's wall-clock time on the original P1 run before being root-caused. Setting this up front avoids it entirely.

In [ ]:
import os
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'

## 5. Probe the T4's actual max batch size, write it into the config

Doubles the batch size until it OOMs, binary-searches the gap, then applies an 85% safety margin (for memory fragmentation over a long run) and writes the result into `configs/p1_augmented_colab_t4max.yaml` in place. Takes under a minute.

In [ ]:
!python -m src.train.find_max_batch_size --config configs/p1_augmented_colab_t4max.yaml --start 16 --max-cap 256

## 6. Launch training

Checkpoints go to `/content/drive/MyDrive/dr_retina_checkpoints/p1_augmented_resnet50_t4max/` (Drive-backed, survives disconnects). If this cell is re-run after a disconnect, flip `RESUME = True` to continue from `last.pt` instead of restarting.

In [ ]:
RESUME = False  # set True after a disconnect/reconnect to continue instead of restarting

In [ ]:
resume_flag = '--resume' if RESUME else ''
!python -m src.train.train_baseline --config configs/p1_augmented_colab_t4max.yaml {resume_flag}

## 7. After training: check GPU memory headroom actually used

Sanity check that the picked batch size was actually using the T4 well (should be well above the ~30-40% utilization the original batch_size=16 run would have gotten).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu --format=csv

## 8. Compare against the original P1 run

Once both finish, pull each run's `history.json` from Drive and compare **internal validation QWK only** (per `docs/SIH_SUBMISSION_PLAN.md`'s selection rule — no peeking at external eval to choose). Use whichever checkpoint wins for `external_eval` and `tune_referable_threshold.py`.

In [ ]:
import json

def best_qwk(history_path):
    history = json.load(open(history_path))
    return max(h['val_qwk'] for h in history)

original = '/content/drive/MyDrive/dr_retina_checkpoints/p1_augmented_resnet50/history.json'
t4max = '/content/drive/MyDrive/dr_retina_checkpoints/p1_augmented_resnet50_t4max/history.json'

import os
for label, path in [('original (batch=16)', original), ('t4max (auto-tuned batch)', t4max)]:
    if os.path.exists(path):
        print(f'{label}: best val QWK = {best_qwk(path):.4f}')
    else:
        print(f'{label}: not finished yet ({path} not found)')